In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import re
import json
import pandas as pd
import time
import datetime
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
# ========== Configuration ==========
API_KEY = os.environ.get("DEEPSEEK_API_KEY")
if not API_KEY:
    API_KEY = "sk-XXXX"   # Replace with your real API key

BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-v4-pro"          # "deepseek-v4-flash", "deepseek-v4-pro"
MAX_WORKERS = 10                    # Number of concurrent threads
REQUESTS_PER_MINUTE = 200          # Rate limit (requests per minute)
MIN_INTERVAL = 60.0 / REQUESTS_PER_MINUTE   # Minimum interval between requests (seconds)

# Price constants (unit: CNY per million tokens)
# deepseek-v4-pro pricing (25% discount before 2026-05-05)
V4_PRO_PRICES = {
    'input_no_cache': {'cache_hit': 0.025, 'cache_miss': 3.0},
    'output': 6.0
}
V4_FLASH_PRICES = {'input': 1.0, 'output': 2.0}

# File paths
INPUT_PKL = r'./data/raw_df.pkl'
OUTPUT_PKL = r'./data/handle_df.pkl'
CHECKPOINT_EVERY = 100              # Save progress every 100 papers
PROMPT_FILE = r'./system_prompt.txt'    # External text file containing the system prompt

# Initialize the synchronous client
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=60)

# Global accumulated cost (unit: CNY)
total_cost = 0.0

# Load the system prompt from an external text file
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()


In [ ]:
# ========== Single-paper scoring function (with retry, cost tracking, rate limiting) ==========
def score_relevance_sync(title, abstract, keywords):
    global total_cost
    user_prompt = f"标题：{title}\n摘要：{abstract}\n关键词：{keywords}"
    
    # Retry strategy: first 3 attempts use JSON Output mode; if it still returns empty content
    # (a known intermittent issue with JSON Output), fall back to normal mode for 2 more attempts
    # (normal mode rarely returns empty content, and the robust parsing below still extracts valid JSON)
    for attempt in range(5):
        use_json_mode = attempt < 3
        try:
            request_start = time.time()
            kwargs = dict(temperature=0, max_tokens=500)
            if use_json_mode:
                kwargs['response_format'] = {'type': 'json_object'}   # Enforce JSON output
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                **kwargs
            )
            execution_time = time.time() - request_start
            
            # Get token usage
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            total_tokens = response.usage.total_tokens
            
            # Calculate the cost of this call
            price_input = price_output = None
            discount_applied = False
            if MODEL == "deepseek-v4-pro":
                is_cache_hit = False   # Simplified: always assume a cache miss; implement cache detection yourself
                discount_deadline = datetime.date(2026, 5, 5)
                is_discount_active = datetime.date.today() <= discount_deadline
                base_input_price = V4_PRO_PRICES['input_no_cache']['cache_hit'] if is_cache_hit else V4_PRO_PRICES['input_no_cache']['cache_miss']
                price_input = base_input_price / 4 if is_discount_active else base_input_price
                price_output = V4_PRO_PRICES['output'] / 4 if is_discount_active else V4_PRO_PRICES['output']
                discount_applied = is_discount_active
            elif MODEL == "deepseek-v4-flash":
                price_input = V4_FLASH_PRICES['input']
                price_output = V4_FLASH_PRICES['output']
            
            if price_input is not None and price_output is not None:
                request_cost = (prompt_tokens / 1_000_000) * price_input + (completion_tokens / 1_000_000) * price_output
                total_cost += request_cost
            
            # Handle empty content (JSON Output intermittently returns empty content, as noted in the official docs)
            raw = response.choices[0].message.content
            if not raw or not raw.strip():
                raise ValueError("API 返回了空内容" + ("（JSON模式，将改用普通模式兜底）" if use_json_mode else ""))
            raw = raw.strip()
            # Strip markdown code fences
            if raw.startswith("```"):
                raw = re.sub(r'^```(?:json)?\s*', '', raw)
                raw = re.sub(r'\s*```$', '', raw)
                raw = raw.strip()
            
            # Try to parse the full JSON directly first (with response_format, the model output is valid JSON)
            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                # Fallback: extract the content between the first { and the last } and parse it
                first_brace = raw.find('{')
                last_brace = raw.rfind('}')
                if first_brace != -1 and last_brace > first_brace:
                    data = json.loads(raw[first_brace:last_brace + 1])
                else:
                    raise ValueError("No valid JSON found")
            
            score = int(data.get("score", 0))
            research_problem = data.get("research_problem") if score >= 6 else None
            research_method = data.get("research_method") if score >= 6 else None
            
            # Rate limiting
            elapsed = time.time() - request_start
            if elapsed < MIN_INTERVAL:
                time.sleep(MIN_INTERVAL - elapsed)
            
            return {"score": score, "research_problem": research_problem, "research_method": research_method}
        
        except Exception as e:
            print(f"⚠️ API调用失败 (尝试 {attempt+1}/5): {e}")
            time.sleep(2 ** (attempt % 3))
    
    # ⭐ All attempts failed: return None (keep the unscored NaN state); the next run will retry automatically
    #    Never return 0 — otherwise an API failure would be misjudged as an "irrelevant paper" and corrupt the data
    return None

# ========== Batch processing function (with checkpoint/resume support) ==========
def process_all_sync(df):
    global total_cost
    # Ensure the columns exist
    if 'relevance_score' not in df.columns:
        df['relevance_score'] = pd.NA
        df['research_problem'] = pd.NA
        df['research_method'] = pd.NA
    else:
        df['relevance_score'] = pd.to_numeric(df['relevance_score'], errors='coerce')
        if 'research_problem' not in df.columns:
            df['research_problem'] = pd.NA
        if 'research_method' not in df.columns:
            df['research_method'] = pd.NA
    
    # Find entries that have not been scored yet (including those that exhausted retries and stayed NaN)
    pending = [(idx, row['Title'], row['Abstract'], row['Keywords']) 
               for idx, row in df.iterrows() if pd.isna(row['relevance_score'])]
    
    if not pending:
        print("🎉 所有文献已完成评分！")
        return df
    
    total = len(pending)
    print(f"📊 需要处理 {total} 篇（总 {len(df)} 篇）")
    print(f"⚙️ 并发线程数={MAX_WORKERS}, 限速={REQUESTS_PER_MINUTE}次/分钟, 每{CHECKPOINT_EVERY}篇保存一次")
    
    results = {}          # idx -> result dict
    completed = 0
    pending_failed = 0    # Number of papers that exhausted retries and remain NaN for the next round
    last_cost = 0.0       # Track the accumulated cost at the last save
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_idx = {
            executor.submit(score_relevance_sync, title, abstract, keywords): idx
            for idx, title, abstract, keywords in pending
        }
        
        with tqdm(total=total, desc="评分进度", unit="篇") as pbar:
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    result = future.result()
                    if result is None:
                        pending_failed += 1   # Keep NaN; do not write 0
                    else:
                        results[idx] = result
                except Exception as e:
                    print(f"\n❌ 未捕获异常 idx={idx}: {e}")
                    pending_failed += 1       # Also keep NaN for retry in the next round
                
                completed += 1
                pbar.update(1)
                
                # Reached a checkpoint
                if completed % CHECKPOINT_EVERY == 0 or completed == total:
                    # Write all current results back to df (None results are skipped and stay NaN for the next round)
                    for i, res in results.items():
                        df.at[i, 'relevance_score'] = res['score']
                        df.at[i, 'research_problem'] = res.get('research_problem')
                        df.at[i, 'research_method'] = res.get('research_method')
                    df.to_pickle(OUTPUT_PKL)
                    
                    # Calculate the cost added in this round
                    batch_cost = total_cost - last_cost
                    last_cost = total_cost
                    print(f"\n💾 已保存 {completed}/{total} 篇")
                    print(f"   📈 本轮花费: ¥{batch_cost:.6f}")
                    print(f"   💰 累计花费: ¥{total_cost:.6f}")
                    pbar.set_postfix({"已保存": completed, "待重试": pending_failed})
    
    # Final save (if the last batch is smaller than CHECKPOINT_EVERY and no save was triggered, save once more here)
    if completed % CHECKPOINT_EVERY != 0:
        for i, res in results.items():
            df.at[i, 'relevance_score'] = res['score']
            df.at[i, 'research_problem'] = res.get('research_problem')
            df.at[i, 'research_method'] = res.get('research_method')
        df.to_pickle(OUTPUT_PKL)
        batch_cost = total_cost - last_cost
        print(f"\n💾 最终保存 {completed}/{total} 篇")
        print(f"   📈 本轮花费: ¥{batch_cost:.6f}")
        print(f"   💰 累计花费: ¥{total_cost:.6f}")
    
    print(f"\n✅ 全部完成！成功 {completed - pending_failed} 篇")
    if pending_failed:
        print(f"⚠️ 有 {pending_failed} 篇重试耗尽，已保持未评分状态，下次运行将自动重试")
    print(f"💾 结果保存至 {OUTPUT_PKL}")
    return df

In [ ]:
# 1. Load the raw data
df = pd.read_pickle(INPUT_PKL)
print(f"原始数据 shape: {df.shape}")

# Optional: test on the first 20 rows (comment out this line for the real run)
# df = df.iloc[:20, :].copy()
# df = df[df['publication_year'] == '2026'].head(50).copy()

# 2. Resume support: if the output file exists, load the existing scores
if os.path.exists(OUTPUT_PKL):
    existing = pd.read_pickle(OUTPUT_PKL)
    # Ensure the columns exist
    for col in ['relevance_score', 'research_problem', 'research_method']:
        if col not in df.columns:
            df[col] = pd.NA
        if col in existing.columns:
            # Align by index (assuming the indexes match)
            df[col] = existing[col]
    print(f"✅ 加载已有进度，已评分 {df['relevance_score'].notna().sum()} 篇")
else:
    # Initialize the new columns
    df['relevance_score'] = pd.NA
    df['research_problem'] = pd.NA
    df['research_method'] = pd.NA
    print("✅ 新建评分表")

# 3. Run the processing
df_final = process_all_sync(df)